# ML Hackathon 2026 — Will this workout be hard for this person?

**Result: 5-fold cross-validation macro-F1 = 0.626, public Kaggle = 0.6523** (this notebook reproduces sub09; the selected submission sub10 is produced by `sub10_stack_locaux.py`, which starts from these predictions).

**Key idea:** `intensity` is the **exact tercile, per (user, sport), of a hidden sensor measure** (probably heart rate), computed over train + test together. Consequences:
1. we compare each workout to **the person's own habits**, not to absolute values;
2. we know **the exact number of easy / moderate / hard workouts of each (user, sport) in the test**, and assign predictions with an optimal assignment (Hungarian algorithm) under these quotas.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                              ExtraTreesClassifier, ExtraTreesRegressor)

DOSSIER = "data" if os.path.exists("data/train.csv") else "."
train = pd.read_csv(f"{DOSSIER}/train.csv")
test = pd.read_csv(f"{DOSSIER}/test.csv")
CLASSES = ["easy", "moderate", "hard"]
y = train["intensity"].map({c: i for i, c in enumerate(CLASSES)}).values
n, nt = len(train), len(test)
print(train.shape, test.shape)

(22728, 18) (7669, 17)


## 1. Understanding the target

The three classes are balanced **within each person**, and raw speed is weakly linked to the target while speed **relative to the person** is much more informative.

In [2]:
part = pd.crosstab(train["user_id"], train["intensity"], normalize="index")
print(part.describe().round(3))
rang = train.groupby("user_id")["speed_kmh"].rank(pct=True)
print("Spearman vitesse brute :", round(pd.Series(y).corr(train["speed_kmh"], method="spearman"), 3))
print("Spearman rang de vitesse intra-personne :", round(pd.Series(y).corr(rang, method="spearman"), 3))

intensity     easy     hard  moderate
count      183.000  183.000   183.000
mean         0.328    0.338     0.334
std          0.040    0.041     0.038
min          0.167    0.200     0.200
25%          0.312    0.322     0.318
50%          0.330    0.334     0.333
75%          0.347    0.352     0.352
max          0.444    0.556     0.464
Spearman vitesse brute : 0.121
Spearman rang de vitesse intra-personne : 0.345


**Quota check.** If the label is an exact tercile over train + test, then `total/3 − train count of the class` is the number of test workouts of that class: never negative, and the sum equals the number of test workouts. We check it per (user, sport).

In [3]:
g_tr = (train["user_id"].astype(str) + "_" + train["sport"]).values
g_te = (test["user_id"].astype(str) + "_" + test["sport"]).values
ct = pd.crosstab(g_tr, train["intensity"])[CLASSES]
nb_test = pd.Series(g_te).value_counts().reindex(ct.index).fillna(0)
quota = ct.rsub((ct.sum(axis=1) + nb_test) / 3, axis=0)
print("quota minimum :", quota.min().round(2).to_dict())
print("somme des quotas == nb séances test :", np.allclose(quota.sum(axis=1), nb_test))

quota minimum : {'easy': 0.0, 'moderate': -0.33, 'hard': -0.67}
somme des quotas == nb séances test : True


## 2. Baselines: 3 model families on raw features

Validation: `StratifiedKFold(5, shuffle=True, random_state=0)`. The test set is a random sample of workouts from the same users, so this split mimics the real situation.

In [4]:
NUM_BRUTES = ["year", "month", "weekday", "start_hour_utc", "duration_min", "distance_km",
              "speed_kmh", "max_speed_kmh", "speed_std_kmh", "altitude_mean_m",
              "altitude_gain_m", "altitude_loss_m", "n_points"]

def nettoyer(df):
    df = df.copy()
    df.loc[df["altitude_mean_m"] < -500, "altitude_mean_m"] = np.nan
    df["altitude_mean_m"] = df["altitude_mean_m"].fillna(df["altitude_mean_m"].median())
    return df

tout = nettoyer(pd.concat([train.drop(columns="intensity"), test], ignore_index=True))
X_brut = pd.concat([tout[NUM_BRUTES], pd.get_dummies(tout["sport"], prefix="sport", dtype=float),
                    pd.get_dummies(tout["gender"], prefix="gender", dtype=float)], axis=1).iloc[:n]
plis = list(StratifiedKFold(5, shuffle=True, random_state=0).split(X_brut, y))

def score_cv(fabrique, X):
    pred = np.zeros(n, dtype=int)
    for a, b in plis:
        pred[b] = fabrique().fit(X.iloc[a], y[a]).predict(X.iloc[b])
    return f1_score(y, pred, average="macro")

baselines = {
    "Régression logistique": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Random forest": lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=0),
    "HistGradientBoosting": lambda: HistGradientBoostingClassifier(random_state=0),
}
for nom, fab in baselines.items():
    print(f"{nom:<25} macro-F1 = {score_cv(fab, X_brut):.4f}")

Régression logistique     macro-F1 = 0.4227


Random forest             macro-F1 = 0.5509


HistGradientBoosting      macro-F1 = 0.5216


## 3. Features relative to the person (no labels)

- derived: continuous time `t`, slope, max/mean speed ratio, speed coefficient of variation, climb per minute;
- for each variable, per user then per (user, sport): percentile rank, z-score, deviation from median;
- user context (means, standard deviations, volume);
- **local features**: deviation from the mean of the same (user, sport) **within ±2 months** (Gaussian kernel), since fitness changes.

Computed on train + test: they use no label, so no leakage.

In [5]:
BASE_REL = ["speed_kmh", "max_speed_kmh", "speed_std_kmh", "duration_min", "distance_km",
            "altitude_gain_m", "altitude_loss_m", "altitude_mean_m", "n_points",
            "pente", "ratio_max", "cv_vitesse", "denivele_min", "t"]
BASE_US = ["speed_kmh", "max_speed_kmh", "speed_std_kmh", "duration_min", "distance_km",
           "altitude_gain_m", "pente", "ratio_max", "t"]


def ajouter_derivees(tout):
    tout = tout.copy()
    tout["t"] = tout["year"] + (tout["month"] - 1) / 12
    tout["pente"] = (tout["altitude_gain_m"] + tout["altitude_loss_m"]) / (tout["distance_km"] * 1000 + 1)
    tout["ratio_max"] = tout["max_speed_kmh"] / (tout["speed_kmh"] + 0.1)
    tout["cv_vitesse"] = tout["speed_std_kmh"] / (tout["speed_kmh"] + 0.1)
    tout["denivele_min"] = tout["altitude_gain_m"] / tout["duration_min"]
    return tout


def relatives(tout, groupe, colonnes, suffixe):
    g = tout.groupby(groupe)
    sortie = {}
    for c in colonnes:
        moy = g[c].transform("mean")
        et = g[c].transform("std").fillna(0) + 1e-6
        med = g[c].transform("median")
        sortie[f"{c}_rg_{suffixe}"] = g[c].rank(pct=True)
        sortie[f"{c}_z_{suffixe}"] = (tout[c] - moy) / et
        sortie[f"{c}_dmed_{suffixe}"] = tout[c] - med
    return pd.DataFrame(sortie, index=tout.index)


def contexte_user(tout):
    g = tout.groupby("user_id")
    ctx = pd.DataFrame(index=tout.index)
    for c in ["speed_kmh", "duration_min", "distance_km", "altitude_gain_m", "t"]:
        ctx[f"user_moy_{c}"] = g[c].transform("mean")
        ctx[f"user_et_{c}"] = g[c].transform("std").fillna(0)
    ctx["user_n"] = g["speed_kmh"].transform("size")
    ctx["user_n_sport"] = tout.groupby(["user_id", "sport"])["speed_kmh"].transform("size")
    return ctx


def construire(tout, avec_user=True, avec_us=True, avec_ctx=True):
    tout = ajouter_derivees(tout)
    morceaux = [tout[["year", "month", "weekday", "start_hour_utc", "duration_min", "distance_km",
                      "speed_kmh", "max_speed_kmh", "speed_std_kmh", "altitude_mean_m",
                      "altitude_gain_m", "altitude_loss_m", "n_points", "t", "pente",
                      "ratio_max", "cv_vitesse", "denivele_min"]]]
    morceaux.append(pd.get_dummies(tout["sport"], prefix="sport", dtype=float))
    morceaux.append(pd.get_dummies(tout["gender"], prefix="gender", dtype=float))
    if avec_user:
        morceaux.append(relatives(tout, "user_id", BASE_REL, "u"))
    if avec_us:
        morceaux.append(relatives(tout, ["user_id", "sport"], BASE_US, "us"))
    if avec_ctx:
        morceaux.append(contexte_user(tout))
    return pd.concat(morceaux, axis=1)

COLS_LOC = ["speed_kmh", "max_speed_kmh", "duration_min", "distance_km", "altitude_gain_m"]


def relatives_locales(tout, sigma=2.0, colonnes=COLS_LOC):
    t = (tout["year"] * 12 + tout["month"] - 1).values.astype(float)
    cle = tout["user_id"].astype(str) + "_" + tout["sport"]
    sortie = np.zeros((len(tout), len(colonnes) * 2 + 1))
    vals = tout[colonnes].values.astype(float)
    for g, idx in tout.groupby(cle).indices.items():
        d = t[idx][:, None] - t[idx][None, :]
        w = np.exp(-0.5 * (d / sigma) ** 2)
        np.fill_diagonal(w, 0)
        sw = w.sum(1, keepdims=True) + 1e-9
        v = vals[idx]
        moy = (w @ v) / sw
        var = (w @ (v ** 2)) / sw - moy ** 2
        et = np.sqrt(np.clip(var, 1e-6, None))
        sortie[idx, :len(colonnes)] = v - moy
        sortie[idx, len(colonnes):2 * len(colonnes)] = (v - moy) / et
        sortie[idx, -1] = sw[:, 0]
    noms = [f"{c}_dloc" for c in colonnes] + [f"{c}_zloc" for c in colonnes] + ["densite_loc"]
    return pd.DataFrame(sortie, columns=noms, index=tout.index)

X = pd.concat([construire(tout), relatives_locales(tout)], axis=1)
tout["vz"] = X["speed_kmh_z_us"]
print("features :", X.shape[1])

features : 120


## 4. Label encodings (computed out-of-fold)

These features use the **known labels** of the same person:
- **current form**: share of easy / hard among the person's workouts, weighted by time proximity (σ = 0.5 / 1.5 / 4 months);
- **time × speed neighbours**: same, weighted by both date gap and relative speed gap;
- **personal model**: vote of the 15 most similar known workouts of the same (user, sport) on 10 dimensions, and distances to the easy / moderate / hard profiles.

**No leakage:** a workout is never encoded with its own label. Training rows of a fold are encoded with a second inner 5-fold split.

In [6]:
SIGMAS = [0.5, 1.5, 4.0]


def temps_mois(df):
    return (df["year"] * 12 + df["month"] - 1).values.astype(float)


def noyau_labels(ref, y_ref, requete, sigmas=SIGMAS):
    sortie = np.zeros((len(requete), len(sigmas) * 3 + 1))
    t_ref = temps_mois(ref)
    t_req = temps_mois(requete)
    u_ref = ref["user_id"].values
    u_req = requete["user_id"].values
    onehot = np.eye(3)[y_ref]
    for u in np.unique(u_req):
        iq = np.where(u_req == u)[0]
        ir = np.where(u_ref == u)[0]
        if len(ir) == 0:
            sortie[iq] = np.nan
            continue
        prior = onehot[ir].mean(0)
        d = t_req[iq][:, None] - t_ref[ir][None, :]
        cols = []
        for s in sigmas:
            w = np.exp(-0.5 * (d / s) ** 2)
            somme = w.sum(1, keepdims=True)
            p = (w @ onehot[ir] + prior) / (somme + 1.0)
            cols += [p[:, 0], p[:, 2], p[:, 2] - p[:, 0]]
        same = (d == 0).sum(1)
        cols.append(same)
        sortie[iq] = np.column_stack(cols)
    noms = []
    for s in sigmas:
        noms += [f"k{s}_easy", f"k{s}_hard", f"k{s}_diff"]
    noms.append("n_meme_mois")
    return pd.DataFrame(sortie, columns=noms, index=requete.index)


def noyau_labels_vitesse(ref, y_ref, requete, s_t=3.0, s_v=0.5):
    sortie = np.full((len(requete), 3), np.nan)
    t_ref, t_req = temps_mois(ref), temps_mois(requete)
    v_ref, v_req = ref["vz"].values, requete["vz"].values
    k_ref = (ref["user_id"].astype(str) + "_" + ref["sport"]).values
    k_req = (requete["user_id"].astype(str) + "_" + requete["sport"]).values
    onehot = np.eye(3)[y_ref]
    for k in np.unique(k_req):
        iq = np.where(k_req == k)[0]
        ir = np.where(k_ref == k)[0]
        if len(ir) == 0:
            continue
        prior = onehot[ir].mean(0)
        dt = (t_req[iq][:, None] - t_ref[ir][None, :]) / s_t
        dv = (v_req[iq][:, None] - v_ref[ir][None, :]) / s_v
        w = np.exp(-0.5 * (dt ** 2 + dv ** 2))
        p = (w @ onehot[ir] + prior) / (w.sum(1, keepdims=True) + 1.0)
        sortie[iq] = np.column_stack([p[:, 0], p[:, 2], p[:, 2] - p[:, 0]])
    return pd.DataFrame(sortie, columns=["kv_easy", "kv_hard", "kv_diff"], index=requete.index)


def tous_encodages(ref, y_ref, requete, avec_vitesse=False):
    morceaux = [noyau_labels(ref, y_ref, requete)]
    if avec_vitesse:
        morceaux.append(noyau_labels_vitesse(ref, y_ref, requete))
    return pd.concat(morceaux, axis=1)


def encodage_oof(df_tr, y_tr, n_splits=5, seed=1, avec_vitesse=False):
    morceaux = []
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=seed)
    for a, b in skf.split(df_tr, y_tr):
        morceaux.append(tous_encodages(df_tr.iloc[a], y_tr[a], df_tr.iloc[b], avec_vitesse))
    return pd.concat(morceaux).loc[df_tr.index]

COLS = ["speed_kmh_z_us", "max_speed_kmh_z_us", "duration_min_z_us", "distance_km_z_us",
        "altitude_gain_m_z_us", "pente_z_us", "ratio_max_z_us", "t_z_us",
        "speed_kmh_zloc", "max_speed_kmh_zloc"]
K = 15


def encoder(ref, y_ref, req, cols=COLS):
    g_ref = (ref["user_id"].astype(str) + "_" + ref["sport"]).values
    g_req = (req["user_id"].astype(str) + "_" + req["sport"]).values
    A, B = ref[cols].values.astype(float), req[cols].values.astype(float)
    A = np.clip(A, -4, 4)
    B = np.clip(B, -4, 4)
    sortie = np.full((len(req), 9), np.nan)
    onehot = np.eye(3)[y_ref]
    for g in np.unique(g_req):
        iq = np.where(g_req == g)[0]
        ir = np.where(g_ref == g)[0]
        if len(ir) < 3:
            continue
        d2 = ((B[iq][:, None, :] - A[ir][None, :, :]) ** 2).sum(-1)
        k = min(K, len(ir))
        voisins = np.argsort(d2, axis=1)[:, :k]
        dv = np.take_along_axis(d2, voisins, 1)
        w = 1.0 / (1.0 + dv)
        vote = (w[:, :, None] * onehot[ir][voisins]).sum(1) / w.sum(1, keepdims=True)
        cols_out = [vote[:, 0], vote[:, 2], vote[:, 2] - vote[:, 0]]
        dist = []
        for c in range(3):
            m = y_ref[ir] == c
            if m.sum() == 0:
                dist.append(np.full(len(iq), np.nan))
            else:
                centre = A[ir][m].mean(0)
                dist.append(np.sqrt(((B[iq] - centre) ** 2).sum(1)))
        dist = np.column_stack(dist)
        cols_out += [dist[:, 0], dist[:, 1], dist[:, 2], dist[:, 0] - dist[:, 2]]
        proj = A[ir][y_ref[ir] == 2].mean(0) - A[ir][y_ref[ir] == 0].mean(0) if (y_ref[ir] == 2).any() and (y_ref[ir] == 0).any() else np.zeros(A.shape[1])
        s_req = B[iq] @ proj
        s_ref = A[ir] @ proj
        rang = (s_ref[None, :] < s_req[:, None]).mean(1)
        cols_out += [s_req, rang]
        sortie[iq] = np.column_stack(cols_out)
    noms = ["pk_easy", "pk_hard", "pk_diff", "pc_d_easy", "pc_d_mod", "pc_d_hard", "pc_d_diff", "pp_score", "pp_rang"]
    return pd.DataFrame(sortie, columns=noms, index=req.index)


def encoder_oof(df, y, seed=7):
    morceaux = []
    for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(df, y):
        morceaux.append(encoder(df.iloc[a], y[a], df.iloc[b]))
    return pd.concat(morceaux).loc[df.index]

## 5. Assignment under quotas (Hungarian algorithm)

For each (user, sport), we know how many workouts of each class are among the unknown workouts. We assign them by minimising the total cost under the quota constraint. When the total is not divisible by 3, the extra workout goes to "hard", the only convention consistent with the data.

In [7]:
CONVENTION = "hard"


def comptes_tertiles(total):
    k, r = divmod(int(total), 3)
    if CONVENTION == "hard":
        return np.array([k, k, k]) + np.array({0: [0, 0, 0], 1: [0, 0, 1], 2: [0, 1, 1]}[r])
    return np.full(3, total / 3)


DELTA = 0.0


def quotas_entiers(n_inconnus, total, comptes_connus):
    cible = comptes_tertiles(total) - comptes_connus
    cible = np.clip(cible, 0, None).astype(float)
    cible = cible * n_inconnus / cible.sum() if cible.sum() > 0 else np.full(3, n_inconnus / 3)
    if DELTA:
        cible = np.clip(cible + np.array([-0.5, 1.0, -0.5]) * DELTA * n_inconnus, 0, None)
        cible = cible * n_inconnus / cible.sum()
    base = np.floor(cible).astype(int)
    reste = n_inconnus - base.sum()
    ordre = np.argsort(-(cible - base))
    base[ordre[:reste]] += 1
    return base


def affecter(proba, users, total_par_user, connus_par_user):
    pred = proba.argmax(1).copy()
    logp = np.log(np.clip(proba, 1e-9, 1))
    for u in np.unique(users):
        idx = np.where(users == u)[0]
        q = quotas_entiers(len(idx), total_par_user[u], connus_par_user[u])
        colonnes = np.repeat(np.arange(3), q)
        cout = -logp[idx][:, colonnes]
        lignes, cols = linear_sum_assignment(cout)
        pred[idx[lignes]] = colonnes[cols]
    return pred


def affecter_cout(cout, users, total_par_user, connus_par_user):
    pred = cout.argmin(1).copy()
    for u in np.unique(users):
        idx = np.where(users == u)[0]
        q = quotas_entiers(len(idx), total_par_user[u], connus_par_user[u])
        colonnes = np.repeat(np.arange(3), q)
        lignes, cols = linear_sum_assignment(cout[idx][:, colonnes])
        pred[idx[lignes]] = colonnes[cols]
    return pred

## 6. "Routine" and "same route" encodings (out-of-fold)

- **routines**: known labels of the same (user, sport) on the same weekday, same time slot, both, and same calendar month;
- **same route**: known workouts of the person on a similar route (distance, mean altitude, climb, descent): speed rank and labels.

In [8]:
GROUPES = {"r_jour": ["weekday"], "r_heure": ["hbin"], "r_jour_heure": ["weekday", "hbin"], "r_mois": ["month"]}
COLS_PARC = ["distance_km", "altitude_mean_m", "altitude_gain_m", "altitude_loss_m"]


def preparer(df):
    df = df.copy()
    df["hbin"] = pd.cut(df["start_hour_utc"], [-1, 5, 9, 12, 15, 18, 21, 24], labels=False).astype(int)
    return df


def coords_parcours(df):
    v = df[COLS_PARC].values.astype(float)
    return np.column_stack([np.log1p(v[:, 0]) / 0.05, v[:, 1] / 20, np.log1p(np.clip(v[:, 2], 0, None)) / 0.3,
                            np.log1p(np.clip(v[:, 3], 0, None)) / 0.3])


def encoder_routine(ref, y_ref, req, lissage=5, k=8):
    ref, req = preparer(ref), preparer(req)
    ref = ref.assign(y=y_ref)
    sortie = pd.DataFrame(index=req.index)
    base = ["user_id", "sport"]
    prior = req[base].merge(ref.groupby(base)["y"].mean().rename("pu"), left_on=base, right_index=True, how="left")["pu"].fillna(1).values
    for nom, extra in GROUPES.items():
        cles = base + extra
        g = ref.groupby(cles)["y"].agg(["sum", "count"])
        m = req[cles].merge(g, left_on=cles, right_index=True, how="left")
        sortie[nom] = (m["sum"].fillna(0).values + lissage * prior) / (m["count"].fillna(0).values + lissage) - prior
        sortie[nom + "_n"] = m["count"].fillna(0).values
    rang = np.full(len(req), np.nan)
    lab = np.full(len(req), np.nan)
    dens = np.zeros(len(req))
    cA, cB = coords_parcours(ref), coords_parcours(req)
    gr = (ref["user_id"].astype(str) + "_" + ref["sport"]).values
    gq = (req["user_id"].astype(str) + "_" + req["sport"]).values
    vr, vq, yr = ref["speed_kmh"].values, req["speed_kmh"].values, ref["y"].values
    for g in np.unique(gq):
        iq = np.where(gq == g)[0]
        ir = np.where(gr == g)[0]
        if len(ir) < 5:
            continue
        d2 = ((cB[iq][:, None, :] - cA[ir][None, :, :]) ** 2).sum(-1)
        kk = min(k, len(ir))
        vois = np.argsort(d2, 1)[:, :kk]
        w = np.exp(-np.take_along_axis(d2, vois, 1) / 4)
        sw = w.sum(1) + 1e-9
        rang[iq] = ((vq[iq][:, None] > vr[ir][vois]) * w).sum(1) / sw
        lab[iq] = (yr[ir][vois] * w).sum(1) / sw
        dens[iq] = sw
    sortie["parc_rang_vitesse"] = rang
    sortie["parc_labels"] = lab
    sortie["parc_densite"] = dens
    return sortie


def encoder_routine_oof(df, y, seed=5):
    morceaux = []
    for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(df, y):
        morceaux.append(encoder_routine(df.iloc[a], y[a], df.iloc[b]))
    return pd.concat(morceaux).loc[df.index]

## 7. Expected speed (no labels)

For each (user, sport), a ridge regression predicts log-speed from log-duration, slope, climb/km, date, season and hour. The **residual** says whether the workout was faster than usual **for this type of workout**: going fast on a long or hilly session is harder.

In [9]:
def residus_attendus(tout, alpha=1.0):
    t = (tout["year"] * 12 + tout["month"]).values.astype(float)
    ld = np.log(tout["duration_min"].values)
    pente = (tout["altitude_gain_m"].values + tout["altitude_loss_m"].values) / (tout["distance_km"].values * 1000 + 1)
    gain_km = tout["altitude_gain_m"].values / (tout["distance_km"].values + 0.1)
    heure = tout["start_hour_utc"].values.astype(float)
    cibles = {"v": np.log(tout["speed_kmh"].values + 0.1), "vmax": np.log(tout["max_speed_kmh"].values + 0.1),
              "vstd": np.log(tout["speed_std_kmh"].values + 0.1)}
    sortie = {f"{k}_res": np.zeros(len(tout)) for k in cibles}
    sortie.update({f"{k}_res_z": np.zeros(len(tout)) for k in cibles})
    sortie["v_res_rang"] = np.zeros(len(tout))
    cle = (tout["user_id"].astype(str) + "_" + tout["sport"]).values
    for g in np.unique(cle):
        idx = np.where(cle == g)[0]
        tc = (t[idx] - t[idx].mean()) / (t[idx].std() + 1)
        X = np.column_stack([np.ones(len(idx)), ld[idx] - ld[idx].mean(), pente[idx], np.log1p(np.clip(gain_km[idx], 0, None)),
                             tc, tc ** 2, np.sin(2 * np.pi * tout["month"].values[idx] / 12), np.cos(2 * np.pi * tout["month"].values[idx] / 12),
                             (heure[idx] - 12) / 12])
        A = X.T @ X + alpha * np.eye(X.shape[1])
        A[0, 0] -= alpha
        for k, v in cibles.items():
            beta = np.linalg.solve(A, X.T @ v[idx])
            r = v[idx] - X @ beta
            sortie[f"{k}_res"][idx] = r
            sortie[f"{k}_res_z"][idx] = r / (r.std() + 1e-6)
        sortie["v_res_rang"][idx] = pd.Series(sortie["v_res"][idx]).rank(pct=True).values
    return pd.DataFrame(sortie, index=tout.index)

ATT = residus_attendus(tout)
att_tr, att_te = ATT.iloc[:n].reset_index(drop=True), ATT.iloc[n:].reset_index(drop=True)

## 8. Pairwise comparison model

A HistGradientBoosting model learns on **pairs of workouts of the same (user, sport)** with different labels to answer "which one is harder?", from the differences between the two workouts. Each workout to predict is then compared to all known workouts of the person, giving its average probability of being harder than the person's known easy, moderate and hard workouts. This is exactly the nature of the target: a ranking within each person.

In [10]:
rng = np.random.default_rng(0)

R = ATT
F = pd.DataFrame({
    "lv": np.log(tout["speed_kmh"] + 0.1), "lvmax": np.log(tout["max_speed_kmh"] + 0.1),
    "lvstd": np.log(tout["speed_std_kmh"] + 0.1), "ldur": np.log(tout["duration_min"]),
    "ldist": np.log(tout["distance_km"] + 0.1),
    "gkm": np.log1p(np.clip(tout["altitude_gain_m"] / (tout["distance_km"] + 0.1), 0, None)),
    "pente": (tout["altitude_gain_m"] + tout["altitude_loss_m"]) / (tout["distance_km"] * 1000 + 1),
    "alt": tout["altitude_mean_m"] / 100, "t": (tout["year"] * 12 + tout["month"]).astype(float),
    "ms": np.sin(2 * np.pi * tout["month"] / 12), "mc": np.cos(2 * np.pi * tout["month"] / 12),
    "h": tout["start_hour_utc"].astype(float), "wd": tout["weekday"].astype(float),
    "vres": R["v_res_z"], "vmres": R["vmax_res_z"],
}).values.astype(np.float32)
G = (tout["user_id"].astype(str) + "_" + tout["sport"]).values
MAXK = 400


def paires_entrainement(idx, yk, par_ligne=12):
    gi = G[idx]
    I, J = [], []
    for g in np.unique(gi):
        loc = np.where(gi == g)[0]
        if len(loc) < 4:
            continue
        for i in loc:
            cand = loc[yk[loc] != yk[i]]
            if len(cand) == 0:
                continue
            j = rng.choice(cand, size=min(par_ligne, len(cand)), replace=False)
            I.append(np.full(len(j), i))
            J.append(j)
    I, J = np.concatenate(I), np.concatenate(J)
    Xp = np.hstack([F[idx[I]] - F[idx[J]], np.abs(F[idx[I], 8:9] - F[idx[J], 8:9])])
    return Xp, (yk[I] > yk[J]).astype(int)


def comparer(modele, idx_k, yk, idx_q):
    out = np.full((len(idx_q), 4), np.nan)
    gk, gq = G[idx_k], G[idx_q]
    for g in np.unique(gq):
        lq = np.where(gq == g)[0]
        lk = np.where(gk == g)[0]
        if len(lk) == 0:
            continue
        if len(lk) > MAXK:
            lk = rng.choice(lk, MAXK, replace=False)
        qi = np.repeat(idx_q[lq], len(lk))
        ki = np.tile(idx_k[lk], len(lq))
        Xp = np.hstack([F[qi] - F[ki], np.abs(F[qi, 8:9] - F[ki, 8:9])])
        p = modele.predict_proba(Xp)[:, 1].reshape(len(lq), len(lk))
        yl = yk[lk]
        cols = [p[:, yl == c].mean(1) if (yl == c).any() else np.full(len(lq), 0.5) for c in range(3)]
        cols.append(p.mean(1))
        out[lq] = np.column_stack(cols)
    return np.nan_to_num(out, nan=0.5)


paires_val = np.zeros((n, 4))
paires_tst = {}
for f, (a, b) in enumerate(plis):
    Xp, yp = paires_entrainement(a, y[a])
    mp = HistGradientBoostingClassifier(learning_rate=0.08, max_iter=300, max_leaf_nodes=31, random_state=0).fit(Xp, yp)
    paires_val[b] = comparer(mp, a, y[a], b)
    paires_tst[f] = comparer(mp, a, y[a], np.arange(n, n + nt))
    print(f"paires fold {f} : {len(yp)} paires")

paires fold 0 : 216184 paires


paires fold 1 : 216010 paires


paires fold 2 : 216037 paires


paires fold 3 : 216064 paires


paires fold 4 : 216209 paires


## 9. Base models per fold, then stacking

For each fold: 6 classifiers (2 HistGradientBoosting + 4 ExtraTrees, with or without routines and expected speed) and 1 ExtraTrees regressor. The test set is predicted by each fold model.

**Stacking:** a logistic regression learns to combine the out-of-fold log-probabilities of the classifiers, the regressor score (and its square) and the 4 pairwise scores. For validation, each fold's meta-model is learned only on the other folds.

**Decision:** for each (user, sport), Hungarian assignment under **exact quotas**.

In [11]:
total = pd.Series(np.concatenate([g_tr, g_te])).value_counts().to_dict()
brut_tr = tout.iloc[:n].reset_index(drop=True)
brut_te = tout.iloc[n:].reset_index(drop=True)
Xtr_all = X.iloc[:n].reset_index(drop=True)
Xte_all = X.iloc[n:].reset_index(drop=True)
cles = ["user_id", "sport"]

def comptes_connus(idx):
    c = pd.crosstab(g_tr[idx], y[idx]).reindex(columns=[0, 1, 2], fill_value=0)
    return {g: c.loc[g].values.astype(float) if g in c.index else np.zeros(3) for g in total}

CLFS = ["hgb_pra", "hgb_pr", "et", "et_r", "et_r_f1", "et_ra_f1"]
val = {k: np.zeros((n, 3)) for k in CLFS}
val["et_reg"] = np.zeros(n)
tst = {f: {} for f in range(5)}
for f, (a, b) in enumerate(plis):
    te_a = encodage_oof(brut_tr.iloc[a], y[a], avec_vitesse=True).reset_index(drop=True)
    te_b = tous_encodages(brut_tr.iloc[a], y[a], brut_tr.iloc[b], avec_vitesse=True).reset_index(drop=True)
    te_t = tous_encodages(brut_tr.iloc[a], y[a], brut_te, avec_vitesse=True).reset_index(drop=True)
    Xa = pd.concat([Xtr_all.iloc[a].reset_index(drop=True), te_a], axis=1).fillna(-1)
    Xb = pd.concat([Xtr_all.iloc[b].reset_index(drop=True), te_b], axis=1).fillna(-1)
    Xt = pd.concat([Xte_all, te_t], axis=1).fillna(-1)
    ref = pd.concat([Xa, train.iloc[a][cles].reset_index(drop=True)], axis=1)
    pa = encoder_oof(ref, y[a]).reset_index(drop=True)
    pb = encoder(ref, y[a], pd.concat([Xb, train.iloc[b][cles].reset_index(drop=True)], axis=1)).reset_index(drop=True)
    pt = encoder(ref, y[a], pd.concat([Xt, test[cles].reset_index(drop=True)], axis=1)).reset_index(drop=True)
    ra = encoder_routine_oof(brut_tr.iloc[a].reset_index(drop=True), y[a]).reset_index(drop=True)
    rb = encoder_routine(brut_tr.iloc[a].reset_index(drop=True), y[a], brut_tr.iloc[b].reset_index(drop=True)).reset_index(drop=True)
    rt = encoder_routine(brut_tr.iloc[a].reset_index(drop=True), y[a], brut_te).reset_index(drop=True)
    jeux = {
        "": (Xa, Xb, Xt),
        "r": tuple(pd.concat([u, v], axis=1).fillna(-1) for u, v in [(Xa, ra), (Xb, rb), (Xt, rt)]),
        "pr": tuple(pd.concat([u, v, w], axis=1).fillna(-1) for u, v, w in [(Xa, pa, ra), (Xb, pb, rb), (Xt, pt, rt)]),
        "pra": tuple(pd.concat([u, v, w, z], axis=1).fillna(-1) for u, v, w, z in [(Xa, pa, ra, att_tr.iloc[a].reset_index(drop=True)), (Xb, pb, rb, att_tr.iloc[b].reset_index(drop=True)), (Xt, pt, rt, att_te)]),
        "ra": tuple(pd.concat([u, w, z], axis=1).fillna(-1) for u, w, z in [(Xa, ra, att_tr.iloc[a].reset_index(drop=True)), (Xb, rb, att_tr.iloc[b].reset_index(drop=True)), (Xt, rt, att_te)]),
    }
    modeles = {
        "hgb_pra": ("pra", HistGradientBoostingClassifier(learning_rate=0.05, max_iter=400, max_leaf_nodes=31, l2_regularization=1.0, random_state=0)),
        "hgb_pr": ("pr", HistGradientBoostingClassifier(learning_rate=0.05, max_iter=400, max_leaf_nodes=31, l2_regularization=1.0, random_state=0)),
        "et": ("", ExtraTreesClassifier(n_estimators=400, min_samples_leaf=3, max_features=0.3, n_jobs=-1, random_state=0)),
        "et_r": ("r", ExtraTreesClassifier(n_estimators=400, min_samples_leaf=3, max_features=0.3, n_jobs=-1, random_state=0)),
        "et_r_f1": ("r", ExtraTreesClassifier(n_estimators=400, min_samples_leaf=1, max_features=0.3, n_jobs=-1, random_state=0)),
        "et_ra_f1": ("ra", ExtraTreesClassifier(n_estimators=400, min_samples_leaf=1, max_features=0.3, n_jobs=-1, random_state=0)),
        "et_reg": ("", ExtraTreesRegressor(n_estimators=400, min_samples_leaf=3, max_features=0.3, n_jobs=-1, random_state=0)),
    }
    for nom, (jeu, m) in modeles.items():
        A, B, T = jeux[jeu]
        m.fit(A, y[a])
        if nom == "et_reg":
            val[nom][b], tst[f][nom] = m.predict(B), m.predict(T)
        else:
            val[nom][b], tst[f][nom] = m.predict_proba(B), m.predict_proba(T)
    print(f"fold {f} : modèles entraînés")

fold 0 : modèles entraînés


fold 1 : modèles entraînés


fold 2 : modèles entraînés


fold 3 : modèles entraînés


fold 4 : modèles entraînés


In [12]:
def bloc(sources):
    cols = [np.log(np.clip(sources[c], 1e-6, 1)) for c in CLFS]
    cols.append(np.column_stack([sources["et_reg"], sources["et_reg"] ** 2]))
    cols.append(sources["paires"])
    return np.hstack(cols)

val["paires"] = paires_val
for f in range(5):
    tst[f]["paires"] = paires_tst[f]
M = bloc(val)
oof = np.zeros(n, dtype=int)
for f, (a, b) in enumerate(plis):
    meta = LogisticRegression(C=1.0, max_iter=3000).fit(M[a], y[a])
    p = np.vstack([meta.predict_proba(M[b]), meta.predict_proba(bloc(tst[f]))])
    pred = affecter_cout(-np.log(np.clip(p, 1e-9, 1)), np.concatenate([g_tr[b], g_te]), total, comptes_connus(a))
    oof[b] = pred[:len(b)]
    print(f"fold {f} : macro-F1 = {f1_score(y[b], oof[b], average='macro'):.4f}")
scores = [f1_score(y[b], oof[b], average="macro") for a, b in plis]
print(f"moyenne = {np.mean(scores):.4f} (écart-type {np.std(scores):.4f})")

fold 0 : macro-F1 = 0.6264


fold 1 : macro-F1 = 0.6343


fold 2 : macro-F1 = 0.6282


fold 3 : macro-F1 = 0.6280


fold 4 : macro-F1 = 0.6137
moyenne = 0.6261 (écart-type 0.0068)


## 10. Error analysis

In [13]:
cm = pd.DataFrame(confusion_matrix(y, oof), index=[f"vrai_{c}" for c in CLASSES],
                  columns=[f"pred_{c}" for c in CLASSES])
print(cm)
print("F1 par classe :", dict(zip(CLASSES, np.round(f1_score(y, oof, average=None), 3))))

               pred_easy  pred_moderate  pred_hard
vrai_easy           5039           1942        548
vrai_moderate       1750           3933       1920
vrai_hard            601           1741       5254
F1 par classe : {'easy': np.float64(0.676), 'moderate': np.float64(0.517), 'hard': np.float64(0.686)}


Errors are almost always between **neighbouring classes**. "moderate" is the most confused class, sitting between the two others. easy ↔ hard confusions are rare. The target is an ordered scale: the model ranks each person's workouts well, the difficulty is placing the two cut-offs.

## 11. Submission

The meta-model is learned on all out-of-fold predictions, then applied to the test predictions of each fold model (average of 5). Assignment uses exact quotas computed with all train labels.

In [14]:
meta = LogisticRegression(C=1.0, max_iter=3000).fit(M, y)
p_test = sum(meta.predict_proba(bloc(tst[f])) for f in range(5)) / 5
pred_test = affecter_cout(-np.log(np.clip(p_test, 1e-9, 1)), g_te, total, comptes_connus(np.arange(n)))
submission = pd.DataFrame({"workout_id": test["workout_id"], "intensity": [CLASSES[i] for i in pred_test]})
submission.to_csv("submission.csv", index=False)
print(submission["intensity"].value_counts(normalize=True).round(3))
submission.head()

intensity
hard        0.338
easy        0.332
moderate    0.330
Name: proportion, dtype: float64


,workout_id,intensity
0,100001,easy
1,100005,moderate
2,100007,easy
3,100008,moderate
4,100009,hard
